# Accumulated geometric defects — section 4.8

Compare ABBA2Midpoint and BM4Midpoint at rho=0.015 for the existing eight periodic candidates and eight chaotic candidates. Cross four fixed steps (20, 40, 80, 160 steps/cycle) with four horizons (10, 20, 40, 80 cycles), always starting at phase zero. Each group has 16 evaluations; each evaluation includes both methods and both diagnostics.

Area is abs(det(M_N)-1), with M_0=I and M_(n+1)=A_n @ M_n along each numerical trajectory. Products use 80 decimal digits and are independently rebuilt at 120 digits; the saved local Jacobians are float64. Reversal is the Euclidean displacement after advancing to a horizon and independently undoing every step with the correct clock. Coordinates stay unwrapped. Raw local Jacobians, accumulated matrices as decimal strings, signed defects, cycle states, returned states, individual timings and input/source fingerprints are saved before reduction.

Completed records are reused by default. The historical 2026-11-18 record remains available. See [the protocol](../../docs/studies/midpoint-geometry-sweep.md).

In [1]:
from pathlib import Path
import json
import sys

# Locate the code repository from this section folder or its parent directories.
ROOT = next(
    (candidate.resolve()
     for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     for candidate in (parent, parent / "GC2D_codigo")
     if (candidate / "src/studies/midpoint_geometry_sweep.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "GC2D_codigo was not found. Keep it beside TFM_redaccion "
        "or start the notebook from the code repository."
    )
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from studies.midpoint_geometry_sweep import RUN_ID, STUDY_PREFIX, COUNTS, HORIZONS
from diagnostics.studies.midpoint_geometry_sweep import verify_record, recover_record

RUN_DIRECTORY = ROOT / "outputs/midpoint_geometry" / RUN_ID
BUCKET = f"gc2d_data:gc2d-notebooks-data/{STUDY_PREFIX}/{RUN_ID}"

RUN_COMPUTATION = False
RECOVER_FROM_BUCKET = True
ARCHIVE_RESULTS = False


## Calculate or recover

A new calculation requires RUN_COMPUTATION=True and a fresh RUN_ID/RUN_DIRECTORY. Scientific results belong in the B2 prefix above; local files are a recovery copy. Timings exclude network transfers. Full computation is explicitly requested research work, not routine notebook validation.

In [2]:
if RUN_COMPUTATION:
    from studies.midpoint_geometry_sweep import run_sweep
    summary = run_sweep(ROOT, RUN_DIRECTORY, counts=COUNTS, horizons=HORIZONS)
else:
    if not (RUN_DIRECTORY / "manifest.json").is_file() and RECOVER_FROM_BUCKET:
        recover_record(BUCKET, RUN_DIRECTORY)
    summary = verify_record(RUN_DIRECTORY)
assert summary["run_id"] == RUN_ID
assert summary["evaluations_per_group"] == 16
assert summary["parameters"]["steps_per_cycle"] == list(COUNTS)
assert summary["parameters"]["horizons"] == list(HORIZONS)
print("Verified:", summary["run_id"], "— 16 evaluations per group")


Verified: midpoint-jacobian-product-20261122-rho0015 — 16 evaluations per group


## Numerical checks and storage

The archive checks reconstruct every ordered matrix product at increased precision and every Euclidean norm from saved raw arrays. Independent oscillator and nonautonomous tests are in tests/test_midpoint_geometry_sweep.py. Set ARCHIVE_RESULTS=True to upload immutable files and verify a fresh read-back. Failure preserves local recoverable results.

In [3]:
checks = json.loads((RUN_DIRECTORY / "checks.json").read_text())
assert checks["status"] == "passed"
print(json.dumps(checks, indent=2))
if ARCHIVE_RESULTS:
    from diagnostics.studies.midpoint_geometry_sweep import archive_record
    print(json.dumps(archive_record(RUN_DIRECTORY), indent=2))
else:
    print("Scientific destination:", BUCKET)


{
  "status": "passed",
  "cases": {
    "ABBA2Midpoint-20.npz": {
      "identity_max_absolute_residual": 6.642517075662949e-16,
      "product_precision_check_digits": 120.0,
      "product_precision_area_max_absolute_difference": 0.0,
      "product_precision_matrix_max_scaled_difference": 8.274185960644919e-76
    },
    "ABBA2Midpoint-40.npz": {
      "identity_max_absolute_residual": 6.055269133331542e-16,
      "product_precision_check_digits": 120.0,
      "product_precision_area_max_absolute_difference": 0.0,
      "product_precision_matrix_max_scaled_difference": 1.751076529862735e-77
    },
    "ABBA2Midpoint-80.npz": {
      "identity_max_absolute_residual": 6.530007277465936e-16,
      "product_precision_check_digits": 120.0,
      "product_precision_area_max_absolute_difference": 0.0,
      "product_precision_matrix_max_scaled_difference": 2.9130608097840774e-77
    },
    "ABBA2Midpoint-160.npz": {
      "identity_max_absolute_residual": 6.05729023335303e-16,
      "prod